In [4]:
import pandas as pd
import numpy as np

# Load dataset and check dimensions
df = pd.read_csv(r'C:\Users\Admin\Desktop\credit_risk_analysis\loan.csv')

print(f"Loaded dataset: {df.shape[0]} rows {df.shape[1]} columns")
print(df.columns.tolist())

C:\Users\Admin\AppData\Local\Temp\ipykernel_11584\4078500003.py:5: DtypeWarning: Columns (0: desc, 1: verification_status_joint) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\Admin\Desktop\credit_risk_analysis\loan.csv')


Loaded dataset: 887379 rows 74 columns
['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m'

In [5]:
# Select target columns for risk analysis
keep_cols = [
    "id",
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "issue_d",
    "loan_status",
    "purpose",
    "dti",
    "addr_state",
    "revol_util",
    "total_pymnt",
]

# Create working subset
df_sub = df[keep_cols].copy()

# Inspect subset structure and data types
print(df_sub.head())
print(df_sub.info())
print(f"Filtered dataset: {df_sub.shape[0]} rows, {df_sub.shape[1]} columns")


        id  loan_amnt        term  int_rate  installment grade sub_grade  \
0  1077501     5000.0   36 months     10.65       162.87     B        B2   
1  1077430     2500.0   60 months     15.27        59.83     C        C4   
2  1077175     2400.0   36 months     15.96        84.33     C        C5   
3  1076863    10000.0   36 months     13.49       339.31     C        C1   
4  1075358     3000.0   60 months     12.69        67.79     B        B5   

  emp_length home_ownership  annual_inc verification_status   issue_d  \
0  10+ years           RENT     24000.0            Verified  Dec-2011   
1   < 1 year           RENT     30000.0     Source Verified  Dec-2011   
2  10+ years           RENT     12252.0        Not Verified  Dec-2011   
3  10+ years           RENT     49200.0     Source Verified  Dec-2011   
4     1 year           RENT     80000.0     Source Verified  Dec-2011   

   loan_status         purpose    dti addr_state  revol_util   total_pymnt  
0   Fully Paid     credit_c

In [ ]:
# Filter completed loans only
df_closed = df_sub[
    df_sub["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])
].copy()

# Create binary target flag (1 = Default, 0 = Fully Paid)
df_closed["is_default"] = (
    df_closed["loan_status"].isin(["Charged Off", "Default"]).astype(int)
)

# Convert percentage strings to numeric values
for col in ["int_rate", "revol_util"]:
    if df_closed[col].dtype == "object":
        df_closed[col] = (
            df_closed[col].astype(str).str.replace("%", "").str.strip()
        )
        df_closed[col] = pd.to_numeric(df_closed[col], errors="coerce")

# Extract numeric loan term in months
df_closed["term_months"] = (
    df_closed["term"].str.extract(r"(\d+)").astype(float)
)

# Map employment length string values to numeric years
emp_map = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10,
}

df_closed["emp_length_years"] = (
    df_closed["emp_length"].astype(str).str.strip().map(emp_map)
)

# Parse issue date and extract temporal attributes
df_closed["issue_date"] = pd.to_datetime(
    df_closed["issue_d"], format="%b-%Y", errors="coerce"
)
df_closed["issue_year"] = df_closed["issue_date"].dt.year
df_closed["issue_month"] = df_closed["issue_date"].dt.month

print(f"Data cleaned and transformed! Total rows: {len(df_closed)}")

emp_length
10+ years    77256
2 years      23647
< 1 year     20975
3 years      20484
5 years      18136
1 year       16951
4 years      16263
6 years      14816
7 years      14156
8 years      11922
NaN           9968
9 years       9616
Name: count, dtype: int64
Data cleaned and transformed! Total rows: 254190


In [17]:
# Calculate monthly gross income
df_closed["monthly_income"] = df_closed["annual_inc"] / 12

# Calculate Payment-to-Income ratio
df_closed["pti"] = df_closed["installment"] / df_closed["monthly_income"]

# Calculate loan principal recovery rate
df_closed["loan_recovery_rate"] = (
    df_closed["total_pymnt"] / df_closed["loan_amnt"]
)

# Segment Debt-to-Income into business risk bands
dti_bins = [-np.inf, 10, 20, 30, np.inf]
dti_labels = [
    "1. Low Risk (<10%)",
    "2. Moderate Risk (10-20%)",
    "3. High Risk (20-30%)",
    "4. Critical Risk (>30%)",
]

df_closed["dti_risk_band"] = pd.cut(
    df_closed["dti"], bins=dti_bins, labels=dti_labels
)

# Inspect newly engineered features
df_closed[
    [
        "annual_inc",
        "monthly_income",
        "installment",
        "pti",
        "dti",
        "dti_risk_band",
        "loan_recovery_rate",
    ]
].head()

,annual_inc,monthly_income,installment,pti,dti,dti_risk_band,loan_recovery_rate
0,24000.0,2000.0,162.87,0.081435,27.65,3. High Risk (20-30%),1.172214
1,30000.0,2500.0,59.83,0.023932,1.00,1. Low Risk (<10%),0.403484
2,12252.0,1021.0,84.33,0.082595,8.72,1. Low Risk (<10%),1.251522
3,49200.0,4100.0,339.31,0.082759,20.00,2. Moderate Risk (10-20%),1.222630
5,36000.0,3000.0,156.46,0.052153,11.20,2. Moderate Risk (10-20%),1.126276


In [18]:
# Save processed dataset to CSV for Power BI import
df_closed.to_csv(
    r"C:\Users\Admin\Desktop\credit_risk_analysis\loan_processed.csv",
    index=False,
)
print("File exported successfully for Power BI!")

File exported successfully for Power BI!
